# 08 — Edge selector grid search

This notebook asks whether there is a small subset of matches where the football model disagrees with the market enough to justify closer tracking.

Important notes:
- API calls: 0.
- Real-money stake: 0.
- This is a paper-only backtest.
- The search uses chronological train / validation / test splits.
- Strategy selection is done on validation and checked on test.

Input:
- `data/processed/07_walkforward_market_backtest/walkforward_eval_dataset.csv`

Output:
- `data/processed/08_edge_selector_grid_report_bundle.zip`


In [ ]:
# Cell 1. Imports and helpers.

from pathlib import Path
import json
import re
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, log_loss

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
IN_DIR = PROCESSED_DIR / "07_walkforward_market_backtest"
OUT_DIR = PROCESSED_DIR / "08_edge_selector_grid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    return path

def multiclass_brier(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    one_hot = np.zeros_like(proba, dtype=float)
    one_hot[np.arange(len(y_true)), y_true] = 1.0
    return float(np.mean(np.sum((proba - one_hot) ** 2, axis=1)))

def evaluate_probs(name, y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    pred = np.argmax(proba, axis=1)
    return {
        "model": name,
        "n": len(y_true),
        "accuracy": float(accuracy_score(y_true, pred)),
        "log_loss": float(log_loss(y_true, proba, labels=[0, 1, 2])),
        "brier": multiclass_brier(y_true, proba),
    }

def safe_div(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return a / b

print("Setup OK.")


In [ ]:
# Cell 2. Load walk-forward evaluation dataset.

candidate_paths = [
    IN_DIR / "walkforward_eval_dataset.csv",
    Path("walkforward_eval_dataset.csv"),
]

eval_path = None
eval_df = pd.DataFrame()

for path in candidate_paths:
    if path.exists():
        eval_df = pd.read_csv(path)
        eval_path = path
        break

if len(eval_df) == 0:
    raise FileNotFoundError(
        "walkforward_eval_dataset.csv not found. "
        "Run 07 first, or copy its output folder to data/processed/."
    )

eval_df["date"] = pd.to_datetime(eval_df["date"], errors="coerce")
eval_df = eval_df.dropna(subset=["date", "outcome"]).copy()
eval_df["outcome"] = eval_df["outcome"].astype(int)
eval_df = eval_df.sort_values("date").reset_index(drop=True)

required = [
    "football_p_away",
    "football_p_draw",
    "football_p_home",
    "T_minus_24h_market_p_away_devig",
    "T_minus_24h_market_p_draw_devig",
    "T_minus_24h_market_p_home_devig",
    "T_minus_24h_best_away_odds",
    "T_minus_24h_best_draw_odds",
    "T_minus_24h_best_home_odds",
]

missing = [c for c in required if c not in eval_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("eval_path:", eval_path)
print("eval_df:", eval_df.shape)
display(eval_df.head())


In [ ]:
# Cell 3. Chronological split.

# We use:
# - train: first 50% for checking baseline;
# - validation: next 25% for choosing alpha/threshold;
# - test: last 25% for honest check.

n = len(eval_df)

q1 = eval_df["date"].quantile(0.50)
q2 = eval_df["date"].quantile(0.75)

eval_df["split"] = np.where(
    eval_df["date"] <= q1,
    "train",
    np.where(eval_df["date"] <= q2, "validation", "test"),
)

split_counts = eval_df.groupby("split").size().reset_index(name="rows")

split_counts.to_csv(
    OUT_DIR / "split_counts.csv",
    index=False,
)

display(split_counts)

if (split_counts["rows"] < 20).any():
    print("WARNING: one split is small. Treat results as exploratory.")


In [ ]:
# Cell 4. Build probability arrays and baseline metrics.

market_cols = [
    "T_minus_24h_market_p_away_devig",
    "T_minus_24h_market_p_draw_devig",
    "T_minus_24h_market_p_home_devig",
]

football_cols = [
    "football_p_away",
    "football_p_draw",
    "football_p_home",
]

baseline_rows = []

for split, part in eval_df.groupby("split"):
    y = part["outcome"].to_numpy()
    market_p = part[market_cols].to_numpy()
    football_p = part[football_cols].to_numpy()

    baseline_rows.append(
        {**evaluate_probs(f"{split}_market_T24", y, market_p),
         "split": split}
    )

    baseline_rows.append(
        {**evaluate_probs(f"{split}_football_walkforward", y, football_p),
         "split": split}
    )

baseline_metrics = pd.DataFrame(baseline_rows)

baseline_metrics.to_csv(
    OUT_DIR / "baseline_metrics_by_split.csv",
    index=False,
)

display(baseline_metrics)


In [ ]:
# Cell 5. Create all possible selection rows.

selection_rows = []

for _, row in eval_df.iterrows():
    market_p = {
        "away": row["T_minus_24h_market_p_away_devig"],
        "draw": row["T_minus_24h_market_p_draw_devig"],
        "home": row["T_minus_24h_market_p_home_devig"],
    }

    football_p = {
        "away": row["football_p_away"],
        "draw": row["football_p_draw"],
        "home": row["football_p_home"],
    }

    odds_24h = {
        "away": row["T_minus_24h_best_away_odds"],
        "draw": row["T_minus_24h_best_draw_odds"],
        "home": row["T_minus_24h_best_home_odds"],
    }

    odds_1h = {
        "away": row.get("T_minus_1h_best_away_odds", np.nan),
        "draw": row.get("T_minus_1h_best_draw_odds", np.nan),
        "home": row.get("T_minus_1h_best_home_odds", np.nan),
    }

    for name, cls in [("away", 0), ("draw", 1), ("home", 2)]:
        selection_rows.append({
            "date": row["date"],
            "split": row["split"],
            "home_team": row["home_team"],
            "away_team": row["away_team"],
            "outcome": int(row["outcome"]),
            "selection": name,
            "selection_class": cls,
            "market_p": market_p[name],
            "football_p": football_p[name],
            "raw_model_market_edge": football_p[name] - market_p[name],
            "best_odds_24h": odds_24h[name],
            "best_odds_1h": odds_1h[name],
            "selection_won": int(int(row["outcome"]) == cls),
        })

selection_df = pd.DataFrame(selection_rows)

selection_df.to_csv(
    OUT_DIR / "all_selection_rows.csv",
    index=False,
)

display(selection_df.head(20))
print("selection rows:", len(selection_df))


In [ ]:
# Cell 6. Strategy simulator.

def simulate_strategy(
    df,
    alpha,
    min_edge,
    min_ev,
    min_odds,
    max_odds,
    require_positive_clv_proxy=False,
):
    # alpha controls how much we trust football model over market.
    # p_blend = market + alpha * (football - market)
    # alpha=0 => market
    # alpha=1 => football

    d = df.copy()

    d["p_blend"] = (
        d["market_p"]
        + alpha * (d["football_p"] - d["market_p"])
    )

    # Keep probabilities sane.
    d["p_blend"] = d["p_blend"].clip(0.001, 0.999)

    d["blend_edge"] = d["p_blend"] - d["market_p"]
    d["blend_ev"] = d["p_blend"] * d["best_odds_24h"] - 1.0

    d["clv_proxy_odds"] = d.apply(
        lambda r: (
            safe_div(r["best_odds_24h"], r["best_odds_1h"]) - 1.0
            if pd.notna(r["best_odds_1h"])
            and r["best_odds_1h"] > 0
            else np.nan
        ),
        axis=1,
    )

    mask = (
        (d["blend_edge"] >= min_edge)
        & (d["blend_ev"] >= min_ev)
        & (d["best_odds_24h"] >= min_odds)
        & (d["best_odds_24h"] <= max_odds)
    )

    if require_positive_clv_proxy:
        mask &= d["clv_proxy_odds"].fillna(-999) > 0

    picks = d[mask].copy()

    if len(picks) == 0:
        return {
            "n_picks": 0,
            "hit_rate": np.nan,
            "roi_flat": np.nan,
            "profit_flat": 0.0,
            "avg_clv_proxy_odds": np.nan,
            "median_clv_proxy_odds": np.nan,
            "avg_blend_ev": np.nan,
            "avg_blend_edge": np.nan,
        }

    picks["profit_1usd"] = np.where(
        picks["selection_won"] == 1,
        picks["best_odds_24h"] - 1.0,
        -1.0,
    )

    return {
        "n_picks": int(len(picks)),
        "hit_rate": float(picks["selection_won"].mean()),
        "roi_flat": float(picks["profit_1usd"].mean()),
        "profit_flat": float(picks["profit_1usd"].sum()),
        "avg_clv_proxy_odds": float(picks["clv_proxy_odds"].mean()),
        "median_clv_proxy_odds": float(picks["clv_proxy_odds"].median()),
        "avg_blend_ev": float(picks["blend_ev"].mean()),
        "avg_blend_edge": float(picks["blend_edge"].mean()),
    }

print("Simulator ready.")


In [ ]:
# Cell 7. Grid search on validation split.

validation = selection_df[selection_df["split"] == "validation"].copy()
test = selection_df[selection_df["split"] == "test"].copy()

alphas = np.round(np.linspace(0.0, 1.0, 21), 2)

min_edges = [
    0.005,
    0.01,
    0.015,
    0.02,
    0.03,
    0.05,
]

min_evs = [
    -0.02,
    0.00,
    0.02,
    0.05,
    0.10,
]

odds_ranges = [
    (1.20, 3.00),
    (1.50, 4.00),
    (1.80, 6.00),
    (2.00, 8.00),
    (1.20, 10.00),
]

require_clv_flags = [False]

rows = []

for alpha in alphas:
    for min_edge in min_edges:
        for min_ev in min_evs:
            for min_odds, max_odds in odds_ranges:
                for require_clv in require_clv_flags:
                    val_stats = simulate_strategy(
                        validation,
                        alpha=alpha,
                        min_edge=min_edge,
                        min_ev=min_ev,
                        min_odds=min_odds,
                        max_odds=max_odds,
                        require_positive_clv_proxy=require_clv,
                    )

                    if val_stats["n_picks"] < 5:
                        continue

                    test_stats = simulate_strategy(
                        test,
                        alpha=alpha,
                        min_edge=min_edge,
                        min_ev=min_ev,
                        min_odds=min_odds,
                        max_odds=max_odds,
                        require_positive_clv_proxy=require_clv,
                    )

                    rows.append({
                        "alpha": alpha,
                        "min_edge": min_edge,
                        "min_ev": min_ev,
                        "min_odds": min_odds,
                        "max_odds": max_odds,
                        "require_positive_clv_proxy": require_clv,
                        **{f"val_{k}": v for k, v in val_stats.items()},
                        **{f"test_{k}": v for k, v in test_stats.items()},
                    })

grid = pd.DataFrame(rows)

if len(grid) > 0:
    # Conservative ranking:
    # prefer enough validation picks, positive validation CLV,
    # positive validation ROI, then check test but do not optimize on test.
    grid = grid.sort_values(
        [
            "val_roi_flat",
            "val_avg_clv_proxy_odds",
            "val_n_picks",
        ],
        ascending=[False, False, False],
    )

grid.to_csv(
    OUT_DIR / "strategy_grid_validation_test.csv",
    index=False,
)

display(grid.head(30))
print("grid rows:", len(grid))


In [ ]:
# Cell 8. Conservative candidate strategies.

if len(grid) == 0:
    candidates = pd.DataFrame()
else:
    candidates = grid[
        (grid["val_n_picks"] >= 8)
        & (grid["val_roi_flat"] > 0)
        & (grid["val_avg_clv_proxy_odds"] > -0.02)
    ].copy()

    # We do not require test profit here, but show it.
    candidates = candidates.sort_values(
        [
            "val_roi_flat",
            "test_avg_clv_proxy_odds",
            "test_roi_flat",
        ],
        ascending=[False, False, False],
    )

candidates.to_csv(
    OUT_DIR / "candidate_strategies.csv",
    index=False,
)

display(candidates.head(20))
print("candidates:", len(candidates))


In [ ]:
# Cell 9. Export picks for best candidate.

if len(candidates) == 0:
    best_params = None
    best_picks_all = pd.DataFrame()
    print("No candidate strategy passed conservative filters.")
else:
    best = candidates.iloc[0].to_dict()

    best_params = {
        "alpha": float(best["alpha"]),
        "min_edge": float(best["min_edge"]),
        "min_ev": float(best["min_ev"]),
        "min_odds": float(best["min_odds"]),
        "max_odds": float(best["max_odds"]),
        "require_positive_clv_proxy": bool(best["require_positive_clv_proxy"]),
    }

    d = selection_df.copy()

    d["p_blend"] = (
        d["market_p"]
        + best_params["alpha"] * (d["football_p"] - d["market_p"])
    ).clip(0.001, 0.999)

    d["blend_edge"] = d["p_blend"] - d["market_p"]
    d["blend_ev"] = d["p_blend"] * d["best_odds_24h"] - 1.0

    d["clv_proxy_odds"] = d.apply(
        lambda r: (
            safe_div(r["best_odds_24h"], r["best_odds_1h"]) - 1.0
            if pd.notna(r["best_odds_1h"])
            and r["best_odds_1h"] > 0
            else np.nan
        ),
        axis=1,
    )

    mask = (
        (d["blend_edge"] >= best_params["min_edge"])
        & (d["blend_ev"] >= best_params["min_ev"])
        & (d["best_odds_24h"] >= best_params["min_odds"])
        & (d["best_odds_24h"] <= best_params["max_odds"])
    )

    if best_params["require_positive_clv_proxy"]:
        mask &= d["clv_proxy_odds"].fillna(-999) > 0

    best_picks_all = d[mask].copy()

    best_picks_all["profit_1usd"] = np.where(
        best_picks_all["selection_won"] == 1,
        best_picks_all["best_odds_24h"] - 1.0,
        -1.0,
    )

    best_picks_all = best_picks_all.sort_values(["date", "home_team"])

best_picks_all.to_csv(
    OUT_DIR / "best_candidate_picks_all_splits.csv",
    index=False,
)

save_json(
    OUT_DIR / "best_candidate_params.json",
    best_params or {},
)

display(best_picks_all.head(50))
print("best picks:", len(best_picks_all))


In [ ]:
# Cell 10. Save report zip.

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "eval_rows": int(len(eval_df)),
    "selection_rows": int(len(selection_df)),
    "grid_rows": int(len(grid)),
    "candidate_rows": int(len(candidates)),
    "best_params": best_params,
    "best_picks": int(len(best_picks_all)),
    "real_money_allowed": False,
    "note": (
        "This is exploratory paper backtest. "
        "Use it to decide next data/modeling steps, not real-money bets."
    ),
}

save_json(
    OUT_DIR / "summary.json",
    summary,
)

zip_path = PROCESSED_DIR / "08_edge_selector_grid_report_bundle.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUT_DIR))

display(pd.DataFrame([summary]))

print("Created:", zip_path.resolve())
print("Report bundle created.")
